# Fine-tune Qwen2.5-1.5B-Instruct bằng Unsloth (LoRA) trên VlogQA
Notebook này hướng dẫn fine-tune mô hình **Qwen2.5-1.5B-Instruct** với **LoRA** qua thư viện **Unsloth** cho tập dữ liệu **VlogQA** (đọc hiểu hội thoại tiếng Việt).

VlogQA là dataset **Extractive QA** (dạng SQuAD): mô hình cần trích xuất một đoạn văn bản ngắn từ context làm câu trả lời.

**Evaluation metrics:** Exact Match (EM) và F1-score.

In [ ]:
# Bước 1: Gỡ cài đặt các gói cũ để tránh xung đột
!pip uninstall torch torchvision torchaudio xformers transformers trl unsloth unsloth_zoo -y

In [1]:
# Bước 2: Cài đặt lại PyTorch với CUDA 12.8
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128 --no-cache-dir

Looking in indexes: https://download.pytorch.org/whl/cu128
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 657.9/657.9 MB 113.3 MB/s  0:00:0600:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 287.2/287.2 MB 112.8 MB/s  0:00:02a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.8/296.8 MB 108.1 MB/s  0:00:02a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.1/139.1 MB 108.1 MB/s  0:00:01a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 113.6 MB/s  0:00:0500:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 150.3 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 112.4 MB/s  0:00:01a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 120.8 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 118.8 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 111.3 MB/s  0:00:00eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 MB 103.1 MB/s 

In [2]:
# Bước 3: Cài đặt transformers, trl, peft, accelerate, bitsandbytes, xformers và unsloth
!pip install transformers trl peft accelerate bitsandbytes xformers
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install unsloth_zoo

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 37.8 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 765.1/765.1 kB 37.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 49.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 40.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.4/842.4 kB 15.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 34.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 65.3 MB/s  0:00:00m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 40.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 31.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 34.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 69.8 MB/s  0:00:00m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.3/801.3 kB 50.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/

## 1. Load Model và Tokenizer với Unsloth

In [1]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 8192  # Tăng từ 1536 → bao phủ P90 VlogQA thực tế (~7052 tokens)
dtype = None           # None để auto-detect (BFloat16 cho 5070 Ti - Blackwell arch)
load_in_4bit = False   # LoRA thuần: nạp model BF16 (~3GB), không quantize

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen2.5-1.5B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

print("Tải model thành công!")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/opt/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.9: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA GeForce RTX 5060 Ti. Num GPUs = 1. Max memory: 15.476 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


unsloth/Qwen2.5-1.5B-Instruct does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
Tải model thành công!


## 2. Cấu hình LoRA Adapter

In [2]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,                          # Rank của LoRA (16 là cân bằng tốt)
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_alpha = 32,                 # Scaling factor (thường = 2 * r)
    lora_dropout = 0.05,             # Dropout nhẹ để tránh overfitting
    bias = "none",
    use_gradient_checkpointing = "unsloth",  # Tiết kiệm VRAM 30%
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

print("Cấu hình LoRA thành công!")

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.6.9 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


Cấu hình LoRA thành công!


## 3. Chuẩn bị Dataset VlogQA

In [3]:
import json
from datasets import Dataset

# ==========================================
# Đọc dữ liệu VlogQA (dạng SQuAD)
# ==========================================
def load_vlogqa(file_path):
    """Đọc file VlogQA và flatten thành danh sách các cặp (context, question, answer)."""
    with open(file_path, "r", encoding="utf-8") as f:
        raw_data = json.load(f)

    samples = []
    for item in raw_data:
        for paragraph in item["paragraphs"]:
            context = paragraph["context"]
            for qa in paragraph["qas"]:
                question = qa["question"]
                # Lấy câu trả lời đầu tiên (có thể có nhiều annotators)
                if qa["answers"]:
                    answer = qa["answers"][0]["text"]
                else:
                    answer = ""  # Bỏ qua câu không có đáp án
                if answer:  # Chỉ lấy mẫu có đáp án
                    samples.append({
                        "context": context,
                        "question": question,
                        "answer": answer,
                        "id": qa.get("id", ""),
                    })
    return samples

# Đường dẫn dữ liệu (điều chỉnh nếu cần)
TRAIN_PATH = "train.json"
DEV_PATH   = "dev.json"
TEST_PATH  = "test.json"

train_samples = load_vlogqa(TRAIN_PATH)
print(f"Tổng số mẫu train: {len(train_samples)}")
print(f"Ví dụ mẫu đầu tiên:")
print(f"  Context (100 ký tự đầu): {train_samples[0]['context'][:100]}")
print(f"  Question: {train_samples[0]['question']}")
print(f"  Answer: {train_samples[0]['answer']}")

Tổng số mẫu train: 8386
Ví dụ mẫu đầu tiên:
  Context (100 ký tự đầu): a cho cô bán anh chị em và các bạn Hôm nay mình khuya sẽ chia sẻ món sườn non sốt bơ tỏi Ừ để mình c
  Question: Nước tương được sử dụng để nêm hãng gì?
  Answer: Maggi


In [4]:
import json
from datasets import Dataset

# ==========================================
# Đọc dữ liệu VlogQA (dạng SQuAD)
# ==========================================
def load_vlogqa(file_path):
    """Đọc file VlogQA và flatten thành danh sách các cặp (context, question, answer)."""
    with open(file_path, "r", encoding="utf-8") as f:
        raw_data = json.load(f)

    samples = []
    for item in raw_data:
        for paragraph in item["paragraphs"]:
            context = paragraph["context"]
            for qa in paragraph["qas"]:
                question = qa["question"]
                if qa["answers"]:
                    answer = qa["answers"][0]["text"]
                else:
                    answer = ""
                if answer:
                    samples.append({
                        "context":  context,
                        "question": question,
                        "answer":   answer,
                        "id":       qa.get("id", ""),
                    })
    return samples


# ==========================================
# Anchor-based Context Truncation
# Cắt context thông minh, ưu tiên giữ answer span
# ==========================================
MAX_CONTEXT_TOKENS = 7500  # Ngân sách context trong cửa sổ 8192 tokens

def truncate_context_around_answer(context, answer, tokenizer, max_ctx_tokens=MAX_CONTEXT_TOKENS):
    """
    Cắt context về max_ctx_tokens tokens, ưu tiên giữ đoạn chứa answer span.
    Đảm bảo answer luôn nằm trong context sau khi cắt.
    """
    answer_start = context.find(answer)
    if answer_start == -1:
        # Không tìm được answer → cắt từ đầu
        ctx_ids = tokenizer.encode(context, add_special_tokens=False)
        if len(ctx_ids) <= max_ctx_tokens:
            return context
        return tokenizer.decode(ctx_ids[:max_ctx_tokens], skip_special_tokens=True)

    # Chia context thành 3 phần: trước / answer / sau
    before_text = context[:answer_start]
    after_text  = context[answer_start + len(answer):]

    before_ids = tokenizer.encode(before_text, add_special_tokens=False)
    answer_ids = tokenizer.encode(answer,       add_special_tokens=False)
    after_ids  = tokenizer.encode(after_text,   add_special_tokens=False)

    # Nếu tổng đã trong ngưỡng → giữ nguyên
    if len(before_ids) + len(answer_ids) + len(after_ids) <= max_ctx_tokens:
        return context

    # Phân bổ budget cho 2 bên
    budget = max_ctx_tokens - len(answer_ids)
    half   = budget // 2

    before_keep = before_ids[-half:]          # Giữ phần cuối của đoạn "trước"
    after_keep  = after_ids[:budget - half]   # Giữ phần đầu của đoạn "sau"

    merged_ids = before_keep + answer_ids + after_keep
    return tokenizer.decode(merged_ids, skip_special_tokens=True)


# ==========================================
# Prompt template cho Extractive QA
# ==========================================
# Prompt mạnh hơn: nhấn mạnh verbatim extraction (đồng bộ với Eval)
SYSTEM_PROMPT = (
    "Bạn là công cụ trích xuất thông tin. "
    "Nhiệm vụ: tìm và COPY NGUYÊN VĂN (từng ký tự) một đoạn ngắn từ đoạn văn được cung cấp để trả lời câu hỏi. "
    "TUYỆT ĐỐI không paraphrase, không thêm bớt từ, không giải thích. "
    "Chỉ trả về đúng chuỗi ký tự xuất hiện trong đoạn văn."
)

def format_prompt_train(context, question, answer, tokenizer):
    """Tạo prompt đầy đủ (cả input lẫn output) để huấn luyện."""
    # Áp dụng anchor truncation trước khi tạo prompt
    context_cropped = truncate_context_around_answer(context, answer, tokenizer)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": (
                f"Đọc đoạn văn sau và trả lời câu hỏi bằng cách trích xuất "
                f"đúng cụm từ xuất hiện trong đoạn văn.\n\n"
                f"Đoạn văn:\n{context_cropped}\n\n"
                f"Câu hỏi: {question}\n\n"
                f"Hãy trả lời ngắn gọn, chỉ là cụm từ trích xuất từ đoạn văn:"
            )
        },
        {"role": "assistant", "content": answer},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)

def format_prompt_inference(context, question, tokenizer):
    """Tạo prompt inference (chỉ input, không có answer)."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": (
                f"Đọc đoạn văn sau và trả lời câu hỏi bằng cách trích xuất "
                f"đúng cụm từ xuất hiện trong đoạn văn.\n\n"
                f"Đoạn văn:\n{context}\n\n"
                f"Câu hỏi: {question}\n\n"
                f"Hãy trả lời ngắn gọn, chỉ là cụm từ trích xuất từ đoạn văn:"
            )
        },
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


# Đường dẫn dữ liệu (điều chỉnh nếu cần)
TRAIN_PATH = "train.json"
DEV_PATH   = "dev.json"
TEST_PATH  = "test.json"

train_samples = load_vlogqa(TRAIN_PATH)
print(f"Tổng số mẫu train: {len(train_samples)}")
print(f"Ví dụ mẫu đầu tiên:")
print(f"  Context (100 ký tự đầu): {train_samples[0]['context'][:100]}")
print(f"  Question: {train_samples[0]['question']}")
print(f"  Answer: {train_samples[0]['answer']}")


# Tạo dataset HuggingFace
def prepare_hf_dataset(samples, tokenizer):
    formatted = []
    for s in samples:
        text = format_prompt_train(s["context"], s["question"], s["answer"], tokenizer)
        formatted.append({"text": text})
    return Dataset.from_list(formatted)

dataset = prepare_hf_dataset(train_samples, tokenizer)
print(f"\nDataset HuggingFace tạo thành công: {len(dataset)} mẫu")
print(f"\n--- Ví dụ prompt đầu tiên ---\n{dataset[0]['text'][:800]}")


Tổng số mẫu train: 8386
Ví dụ mẫu đầu tiên:
  Context (100 ký tự đầu): a cho cô bán anh chị em và các bạn Hôm nay mình khuya sẽ chia sẻ món sườn non sốt bơ tỏi Ừ để mình c
  Question: Nước tương được sử dụng để nêm hãng gì?
  Answer: Maggi

Dataset HuggingFace tạo thành công: 8386 mẫu

--- Ví dụ prompt đầu tiên ---
<|im_start|>system
Bạn là một trợ lý AI thông minh, giỏi tiếng Việt. Nhiệm vụ của bạn là đọc đoạn văn và trả lời câu hỏi bằng cách trích xuất chính xác đoạn văn bản ngắn nhất từ ngữ cảnh được cung cấp.<|im_end|>
<|im_start|>user
Đọc đoạn văn sau và trả lời câu hỏi bằng cách trích xuất đúng cụm từ xuất hiện trong đoạn văn.

Đoạn văn:
a cho cô bán anh chị em và các bạn Hôm nay mình khuya sẽ chia sẻ món sườn non sốt bơ tỏi Ừ để mình có thịt sườn heo nên mua cái sườn non ý là cái xương này tức là khi mình chọn sườn mình nên mua cái sườn mà có cái xương nhỏ hay gãy xương là một cái tơ để lên cho một số các chị em mình lựa sườn thì nhớ cái cây sườn với phương này mới nhỏ con nếu n

## 4. Cấu hình và Bắt đầu Huấn luyện (Fine-Tuning)

In [5]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported
import sys

trainer = SFTTrainer(
    model = model,
    processing_class = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,          # 8192
    dataset_num_proc = 1,      # Dùng 1 process để tránh lỗi pickle
    packing = True,            # Tận dụng GPU: pack nhiều mẫu ngắn thành 1 sequence 8192
    args = TrainingArguments(
        per_device_train_batch_size = 4,      # Tăng từ 2 → 4 (15GB VRAM dư sức)
        gradient_accumulation_steps = 4,      # Effective batch = 4 * 4 = 16
        warmup_steps = 50,                    # Tăng từ 10 → 50 (ổn định với batch lớn hơn)
        num_train_epochs = 3,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),       # 5070 Ti hỗ trợ BFloat16
        logging_steps = 20,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "outputs_vlogqa",
        save_strategy = "epoch",
        report_to = "none",
    ),
)

# Fix sys.modules để tránh lỗi pickle với unsloth
real_config_cls = type(trainer.args)
real_trainer_cls = type(trainer)
sys.modules['trl.trainer.sft_config'] = sys.modules[real_config_cls.__module__]
sys.modules['trl.trainer.sft_trainer'] = sys.modules[real_trainer_cls.__module__]
sys.modules[real_config_cls.__module__].SFTConfig = real_config_cls
sys.modules[real_trainer_cls.__module__].SFTTrainer = real_trainer_cls

# Bắt đầu train!
print("\nBắt đầu fine-tuning...")
trainer_stats = trainer.train()


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.



Bắt đầu fine-tuning...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 8,386 | Num Epochs = 3 | Total steps = 1,575
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
20,3.482578
40,3.171962
60,3.021032
80,2.922239
100,2.871894
120,2.855692
140,2.763390
160,2.751849
180,2.736954
200,2.667301


Unsloth: Restored added_tokens_decoder metadata in outputs_vlogqa/checkpoint-525/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs_vlogqa/checkpoint-1050/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs_vlogqa/checkpoint-1575/tokenizer_config.json.


## 5. Kiểm tra Inference nhanh sau khi huấn luyện

In [6]:
# Kích hoạt chế độ sinh token nhanh của Unsloth (2x tốc độ)
FastLanguageModel.for_inference(model)

# Thử nghiệm với mẫu đầu tiên trong tập train
sample = train_samples[0]
prompt = format_prompt_inference(sample["context"], sample["question"], tokenizer)

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
outputs = model.generate(
    input_ids = inputs["input_ids"],
    attention_mask = inputs["attention_mask"],
    max_new_tokens = 64,
    use_cache = True,
    do_sample = False,    # Greedy decoding cho QA
    temperature = 1.0,
)

input_length = inputs["input_ids"].shape[1]
generated_tokens = outputs[0][input_length:]
response = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

print("\n--- KẾT QUẢ INFERENCE ---")
print(f"Câu hỏi : {sample['question']}")
print(f"Đáp án đúng : {sample['answer']}")
print(f"Model trả lời: {response}")

Both `max_new_tokens` (=64) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/opt/venv/lib/python3.12/site-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/opt/venv/lib/python3.12/site-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/opt/venv/lib/python3.12/sit


--- KẾT QUẢ INFERENCE ---
Câu hỏi : Nước tương được sử dụng để nêm hãng gì?
Đáp án đúng : Maggi
Model trả lời: nước tương Maggi


## 6. Lưu Model LoRA

In [7]:
# Chỉ lưu LoRA weights (nhẹ, vài chục MB)
LORA_SAVE_PATH = "qwen2.5-1.5b-instruct-lora-vlogqa"

model.save_pretrained(LORA_SAVE_PATH)
tokenizer.save_pretrained(LORA_SAVE_PATH)

print(f"Đã lưu thành công mô hình LoRA tại: {LORA_SAVE_PATH}")

# Tùy chọn: Merge LoRA vào model gốc và lưu dạng full weights
# model.save_pretrained_merged("qwen2.5-1.5b-instruct-vlogqa-merged", tokenizer, save_method="merged_16bit")

Unsloth: Restored added_tokens_decoder metadata in qwen2.5-1.5b-instruct-lora-vlogqa/tokenizer_config.json.


Đã lưu thành công mô hình LoRA tại: qwen2.5-1.5b-instruct-lora-vlogqa


---
## 7. Đánh giá (Evaluation) trên tập Test
### Metrics: Exact Match (EM) và F1-score

- **Exact Match (EM):** Tỷ lệ câu trả lời của model khớp **hoàn toàn** với đáp án (sau khi chuẩn hóa text).
- **F1-score:** Đo lường overlap từ-vựng giữa dự đoán và đáp án (tính theo cấp độ token).

> **Lưu ý:** Cell này có thể chạy trong một session mới sau khi đã train và lưu model.

In [9]:
import json
import re
import string
import unicodedata
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from collections import Counter

# ==========================================
# CẤU HÌNH
# ==========================================
BASE_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
ADAPTER_PATH    = "qwen2.5-1.5b-instruct-lora-vlogqa"   # Đường dẫn LoRA đã lưu
TEST_DATA_PATH  = "test.json"
MAX_EVAL_TOKENS = 7500   # Ngân sách context khi inference (giống training)
# Prompt mạnh hơn: nhấn mạnh verbatim extraction
SYSTEM_PROMPT = (
    "Bạn là công cụ trích xuất thông tin. "
    "Nhiệm vụ: tìm và COPY NGUYÊN VĂN (từng ký tự) một đoạn ngắn từ đoạn văn được cung cấp để trả lời câu hỏi. "
    "TUYỆT ĐỐI không paraphrase, không thêm bớt từ, không giải thích. "
    "Chỉ trả về đúng chuỗi ký tự xuất hiện trong đoạn văn."
)

In [10]:
# ==========================================
# LOAD MODEL ĐÃ FINE-TUNE
# ==========================================
print("Loading Tokenizer và Model...")
tokenizer_eval = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)
base_model_eval = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_NAME,
    torch_dtype = torch.float16,
    device_map = "auto"
)
model_eval = PeftModel.from_pretrained(base_model_eval, ADAPTER_PATH)
model_eval.eval()
print("Load model hoàn tất!")

Loading Tokenizer và Model...


Load model hoàn tất!


In [11]:
# ==========================================
# HÀM TIỆN ÍCH: CHUẨN HÓA VĂN BẢN
# ==========================================
def normalize_text(text: str) -> str:
    text = unicodedata.normalize("NFC", text)
    text = text.lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = " ".join(text.split())
    return text


def compute_exact_match(prediction: str, ground_truth: str) -> int:
    return int(normalize_text(prediction) == normalize_text(ground_truth))


def compute_f1_token(prediction: str, ground_truth: str) -> float:
    pred_tokens = normalize_text(prediction).split()
    true_tokens = normalize_text(ground_truth).split()
    if len(pred_tokens) == 0 and len(true_tokens) == 0:
        return 1.0
    if len(pred_tokens) == 0 or len(true_tokens) == 0:
        return 0.0
    common = Counter(pred_tokens) & Counter(true_tokens)
    num_common = sum(common.values())
    if num_common == 0:
        return 0.0
    precision = num_common / len(pred_tokens)
    recall    = num_common / len(true_tokens)
    return (2 * precision * recall) / (precision + recall)


def get_best_score(prediction: str, answers: list) -> tuple:
    em_scores = [compute_exact_match(prediction, ans["text"]) for ans in answers]
    f1_scores = [compute_f1_token(prediction, ans["text"]) for ans in answers]
    return max(em_scores), max(f1_scores)


# ==========================================
# Best-Span Post-processing
# Tìm substring trong context khớp nhất với prediction của model
# ==========================================
def find_best_span(prediction: str, context: str) -> str:
    """
    Sau khi model sinh ra text, tìm substring trong context
    có token-F1 cao nhất so với prediction.
    Rất hiệu quả khi model paraphrase thay vì copy nguyên văn.
    """
    pred_norm = normalize_text(prediction)
    if not pred_norm:
        return prediction

    pred_words = pred_norm.split()
    ctx_norm   = normalize_text(context)
    ctx_words  = ctx_norm.split()
    ctx_orig   = context.split()   # giữ nguyên văn để reconstruct

    n = len(pred_words)
    if n == 0 or len(ctx_words) == 0:
        return prediction

    # Tính F1 của raw prediction với context-normalized để làm baseline
    baseline_f1 = compute_f1_token(pred_norm, ctx_norm)
    best_f1     = baseline_f1
    best_span   = prediction

    # Slide window kích thước n//2 đến 2n+5 qua context
    min_w = max(1, n // 2)
    max_w = min(2 * n + 5, len(ctx_words))

    for w in range(min_w, max_w + 1):
        for i in range(len(ctx_words) - w + 1):
            span_norm  = " ".join(ctx_words[i:i + w])
            f1 = compute_f1_token(pred_norm, span_norm)
            if f1 > best_f1:
                best_f1  = f1
                best_span = " ".join(ctx_orig[i:i + w])

    return best_span


# ==========================================
# HÀM TẠO PROMPT INFERENCE (có pre-truncate context)
# ==========================================
def truncate_context_for_eval(context, tokenizer, max_ctx_tokens=MAX_EVAL_TOKENS):
    """
    Pre-truncate context trước khi đưa vào prompt,
    đảm bảo question LUÔN xuất hiện trong prompt.
    Giữ phần đầu context (heuristic: câu trả lời phân bố đều).
    """
    ctx_ids = tokenizer.encode(context, add_special_tokens=False)
    if len(ctx_ids) <= max_ctx_tokens:
        return context
    return tokenizer.decode(ctx_ids[:max_ctx_tokens], skip_special_tokens=True)


def format_prompt_inference_eval(context, question, tokenizer):
    """Tạo prompt cho inference với pre-truncated context."""
    # Pre-truncate để đảm bảo question không bị đẩy ra ngoài 8192 tokens
    context_cropped = truncate_context_for_eval(context, tokenizer)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": (
                f"Đoạn văn:\n{context_cropped}\n\n"
                f"Câu hỏi: {question}\n\n"
                f"Trích xuất nguyên văn từ đoạn văn (không thêm bớt):"
            )
        },
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


# ==========================================
# ĐỌC DỮ LIỆU TEST
# ==========================================
def load_vlogqa_with_all_answers(file_path):
    """Đọc VlogQA và giữ nguyên tất cả các đáp án để tính best-of-N metric."""
    with open(file_path, "r", encoding="utf-8") as f:
        raw_data = json.load(f)

    samples = []
    for item in raw_data:
        for paragraph in item["paragraphs"]:
            context = paragraph["context"]
            for qa in paragraph["qas"]:
                if qa["answers"]:
                    samples.append({
                        "id": qa.get("id", ""),
                        "context": context,
                        "question": qa["question"],
                        "answers": qa["answers"],
                    })
    return samples

test_samples = load_vlogqa_with_all_answers(TEST_DATA_PATH)
print(f"Tổng số câu hỏi test: {len(test_samples)}")
print("Các hàm metric và best-span đã sẵn sàng!")

Tổng số câu hỏi test: 1062
Các hàm metric và best-span đã sẵn sàng!


In [4]:
# ==========================================
# HÀM TẠO PROMPT INFERENCE
# ==========================================
def format_prompt_inference_eval(context, question, tokenizer):
    """Tạo prompt cho inference (không có đáp án)."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": (
                f"Đọc đoạn văn sau và trả lời câu hỏi bằng cách trích xuất đúng cụm từ xuất hiện trong đoạn văn.\\n\\n"
                f"Đoạn văn:\\n{context}\\n\\n"
                f"Câu hỏi: {question}\\n\\n"
                f"Hãy trả lời ngắn gọn, chỉ là cụm từ trích xuất từ đoạn văn:"
            )
        },
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return text


# ==========================================
# ĐỌC DỮ LIỆU TEST
# ==========================================
def load_vlogqa_with_all_answers(file_path):
    """Đọc VlogQA và giữ nguyên tất cả các đáp án để tính best-of-N metric."""
    with open(file_path, "r", encoding="utf-8") as f:
        raw_data = json.load(f)

    samples = []
    for item in raw_data:
        for paragraph in item["paragraphs"]:
            context = paragraph["context"]
            for qa in paragraph["qas"]:
                if qa["answers"]:
                    samples.append({
                        "id": qa.get("id", ""),
                        "context": context,
                        "question": qa["question"],
                        "answers": qa["answers"],   # Giữ toàn bộ danh sách đáp án
                    })
    return samples

test_samples = load_vlogqa_with_all_answers(TEST_DATA_PATH)
print(f"Tổng số câu hỏi test: {len(test_samples)}")


Tổng số câu hỏi test: 1062


In [12]:
# ==========================================
# CHẠY INFERENCE TRÊN TOÀN BỘ TẬP TEST
# ==========================================
all_em_raw      = []   # EM không post-process
all_f1_raw      = []   # F1 không post-process
all_em_span     = []   # EM sau best-span matching
all_f1_span     = []   # F1 sau best-span matching
all_predictions = []

print(f"\nBắt đầu đánh giá trên {len(test_samples)} câu hỏi...\n")

for sample in tqdm(test_samples, desc="Evaluating"):
    prompt = format_prompt_inference_eval(
        context   = sample["context"],
        question  = sample["question"],
        tokenizer = tokenizer_eval
    )

    # Tokenize (context đã được pre-truncate trong format_prompt_inference_eval)
    inputs = tokenizer_eval(
        prompt,
        return_tensors = "pt",
        truncation = True,
        max_length = 8192,
    ).to(model_eval.device)

    # Generate
    with torch.no_grad():
        outputs = model_eval.generate(
            **inputs,
            max_new_tokens = 64,           # tăng từ 50 → 64 cho an toàn
            do_sample = False,
            temperature = 1.0,
            pad_token_id = tokenizer_eval.pad_token_id,
            eos_token_id = tokenizer_eval.eos_token_id,
        )

    # Decode raw prediction
    input_length   = inputs["input_ids"].shape[1]
    generated_ids  = outputs[0][input_length:]
    raw_prediction = tokenizer_eval.decode(generated_ids, skip_special_tokens=True).strip()
    raw_prediction = raw_prediction.split("\n")[0].strip()

    # Post-processing: snap về span trong context gốc
    span_prediction = find_best_span(raw_prediction, sample["context"])

    # Tính metric cho cả raw và span
    em_raw,  f1_raw  = get_best_score(raw_prediction,  sample["answers"])
    em_span, f1_span = get_best_score(span_prediction, sample["answers"])

    all_em_raw.append(em_raw);   all_f1_raw.append(f1_raw)
    all_em_span.append(em_span); all_f1_span.append(f1_span)
    all_predictions.append({
        "id":             sample["id"],
        "question":       sample["question"],
        "ground_truth":   sample["answers"][0]["text"],
        "raw_prediction":  raw_prediction,
        "span_prediction": span_prediction,
        "em_raw":  em_raw,  "f1_raw":  f1_raw,
        "em_span": em_span, "f1_span": f1_span,
    })

    tqdm.write(
        f"[{len(all_predictions)}/{len(test_samples)}] "
        f"Q: {sample['question'][:40]}... | "
        f"Truth: {sample['answers'][0]['text'][:30]} | "
        f"Raw: {raw_prediction[:30]} | "
        f"Span: {span_prediction[:30]} | "
        f"EM_raw={em_raw} EM_span={em_span} "
        f"F1_raw={f1_raw:.3f} F1_span={f1_span:.3f}"
    )

print("\nHoàn tất inference!")


Bắt đầu đánh giá trên 1062 câu hỏi...

[1/1062] Q: Lò được làm nóng trước khi nướng bao lâu... | Truth: 10 phút | Raw: 10 phút | Span: 10 phút | EM_raw=1 EM_span=1 F1_raw=1.000 F1_span=1.000
[2/1062] Q: Đầu bếp sử dụng gì để trang trí bánh?... | Truth: mấy cái hoa hồi | Raw: hoi | Span: hoi | EM_raw=0 EM_span=0 F1_raw=0.000 F1_span=0.000
[3/1062] Q: Bột ca cao được hòa tan bằng gì?... | Truth: nước ấm | Raw: nước cốt dừa | Span: nước cốt dừa | EM_raw=0 EM_span=0 F1_raw=0.400 F1_span=0.400
[4/1062] Q: Việc làm nóng lò trước khi nướng để làm ... | Truth: tạo rễ tre cho bánh và để bánh | Raw: chị nướng | Span: nướng | EM_raw=0 EM_span=0 F1_raw=0.000 F1_span=0.000
[5/1062] Q: Nếu để hỗn hợp ca cao sôi mạnh thì sẽ mấ... | Truth: nước cốt dừa | Raw: dường di động | Span: dường di động | EM_raw=0 EM_span=0 F1_raw=0.000 F1_span=0.000
[6/1062] Q: Nếu sử dụng trứng vịt thì bánh flan sẽ n... | Truth: ngon hơn | Raw: món bình thường Còn egg flocon | Span: các bạn thường nói là rụng trứ | EM_raw=0

In [14]:
# ==========================================
# IN KẾT QUẢ ĐÁNH GIÁ
# ==========================================
total = len(all_em_raw)

em_raw_score   = sum(all_em_raw)  / total * 100
f1_raw_score   = sum(all_f1_raw)  / total * 100
em_span_score  = sum(all_em_span) / total * 100
f1_span_score  = sum(all_f1_span) / total * 100

print("\n" + "=" * 60)
print("   KẾT QUẢ ĐÁNH GIÁ TRÊN TẬP TEST - VlogQA")
print("   Model: Qwen2.5-1.5B-Instruct + LoRA (Unsloth)")
print("=" * 60)
print(f"  Tổng số câu hỏi test: {total}")
print()
print(f"  {'Metric':<25} {'Raw (no post-proc)':>20} {'Best-Span':>20}")
print(f"  {'-'*65}")
print(f"  {'Exact Match (EM)':<25} {em_raw_score:>19.2f}% {em_span_score:>19.2f}%")
print(f"  {'F1-score (token)':<25} {f1_raw_score:>19.2f}% {f1_span_score:>19.2f}%")
print("=" * 60)

# Phân phối F1 (span version)
f1_buckets = {"0-0.2": 0, "0.2-0.5": 0, "0.5-0.8": 0, "0.8-1.0": 0, "1.0": 0}
for f1 in all_f1_span:
    if f1 == 1.0:    f1_buckets["1.0"] += 1
    elif f1 >= 0.8:  f1_buckets["0.8-1.0"] += 1
    elif f1 >= 0.5:  f1_buckets["0.5-0.8"] += 1
    elif f1 >= 0.2:  f1_buckets["0.2-0.5"] += 1
    else:            f1_buckets["0-0.2"] += 1

print(f"\n  EM đúng (span): {sum(all_em_span)}/{total} câu")
print("\n  Phân phối F1-score (Best-Span):")
for bucket, count in f1_buckets.items():
    print(f"    F1 = {bucket}: {count} câu ({count/total*100:.1f}%)")

# Lưu kết quả
with open("eval_results_vlogqa.json", "w", encoding="utf-8") as f:
    json.dump({
        "em_raw":    round(em_raw_score,  4),
        "f1_raw":    round(f1_raw_score,  4),
        "em_span":   round(em_span_score, 4),
        "f1_span":   round(f1_span_score, 4),
        "total":     total,
        "predictions": all_predictions
    }, f, ensure_ascii=False, indent=2)

print("\nKết quả chi tiết đã được lưu tại: eval_results_vlogqa.json")


   KẾT QUẢ ĐÁNH GIÁ TRÊN TẬP TEST - VlogQA
   Model: Qwen2.5-1.5B-Instruct + LoRA (Unsloth)
  Tổng số câu hỏi test: 1062

  Metric                      Raw (no post-proc)            Best-Span
  -----------------------------------------------------------------
  Exact Match (EM)                        16.38%               14.69%
  F1-score (token)                        30.26%               28.56%

  EM đúng (span): 156/1062 câu

  Phân phối F1-score (Best-Span):
    F1 = 0-0.2: 643 câu (60.5%)
    F1 = 0.2-0.5: 96 câu (9.0%)
    F1 = 0.5-0.8: 137 câu (12.9%)
    F1 = 0.8-1.0: 30 câu (2.8%)
    F1 = 1.0: 156 câu (14.7%)

Kết quả chi tiết đã được lưu tại: eval_results_vlogqa.json


In [7]:
# ==========================================
# XEM CÁC MẪU SAI ĐỂ PHÂN TÍCH
# ==========================================
wrong_preds = [p for p in all_predictions if p["em"] == 0]
print(f"\nSố câu EM = 0 (sai hoàn toàn): {len(wrong_preds)}")
print("\n--- 5 mẫu sai đầu tiên ---")
for p in wrong_preds[:5]:
    print(f"\n  Q: {p['question']}")
    print(f"  Ground truth: {p['ground_truth']}")
    print(f"  Prediction  : {p['prediction']}")
    print(f"  F1          : {p['f1']:.4f}")

# Lưu kết quả ra file JSON để phân tích sau
with open("eval_results_vlogqa.json", "w", encoding="utf-8") as f:
    json.dump({
        "exact_match": round(exact_match_score, 4),
        "f1_score": round(f1_score_avg, 4),
        "total": total,
        "predictions": all_predictions
    }, f, ensure_ascii=False, indent=2)

print("\nKết quả chi tiết đã được lưu tại: eval_results_vlogqa.json")


Số câu EM = 0 (sai hoàn toàn): 905

--- 5 mẫu sai đầu tiên ---

  Q: Đầu bếp sử dụng gì để trang trí bánh?
  Ground truth: mấy cái hoa hồi
  Prediction  : hóa khóc và hic hic hihi
  F1          : 0.0000

  Q: Bột ca cao được hòa tan bằng gì?
  Ground truth: nước ấm
  Prediction  : nước cốt dừa
  F1          : 0.4000

  Q: Việc làm nóng lò trước khi nướng để làm gì?
  Ground truth: tạo rễ tre cho bánh và để bánh được chín đều
  Prediction  : diệt gốc trên bề mặt
  F1          : 0.0000

  Q: Nếu để hỗn hợp ca cao sôi mạnh thì sẽ mất đi vị ngon của nguyên liệu nào?
  Ground truth: nước cốt dừa
  Prediction  : hở
  F1          : 0.0000

  Q: Nếu sử dụng trứng vịt thì bánh flan sẽ như thế nào so với trứng gà?
  Ground truth: ngon hơn
  Prediction  : xanh hơn
  F1          : 0.5000

Kết quả chi tiết đã được lưu tại: eval_results_vlogqa.json
